# Retirement Planner

A self-contained model for projecting your retirement portfolio,
estimating *when* you can retire, and stress-testing that plan against
market variability.

Use the **sidebar** to adjust your assumptions. The deterministic projection
and earliest retirement age update live as you change any input.
Click **Run Monte Carlo** when you want the randomised stress-test results.

**How returns work:** Enter the nominal rates of return you see on your
brokerage or financial website (e.g. 10%). Set your expected inflation rate
once — the model converts to real returns internally and displays all
projected balances in today's purchasing power.

All dollar amounts (expenses, savings, Social Security) should be entered
in today's dollars.

This is a planning aid, not financial advice. See the **Notes & Caveats**
section at the end for the simplifications baked into this model.


In [ ]:
%matplotlib inline
import json
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Layout, AppLayout
from IPython.display import display

matplotlib.rcParams["figure.dpi"] = 110
pd.options.display.float_format = lambda x: f"{x:,.0f}"

In [ ]:
def _spending_curve(base_expenses, age, a):
    """Apply the selected spending model to return age-adjusted expenses."""
    model = a.get("spending_model", "flat")
    p = a.get("spending_params", {})
    if model == "three_phase":
        if age >= p.get("no_go_age", 85):
            return base_expenses * p.get("no_go_pct", 0.60)
        elif age >= p.get("slow_go_age", 75):
            return base_expenses * p.get("slow_go_pct", 0.80)
        return base_expenses
    elif model == "taper":
        start = p.get("taper_start_age", 75)
        if age >= start:
            return base_expenses * (1 - p.get("taper_rate", 0.015)) ** (age - start)
        return base_expenses
    return base_expenses


def project_portfolio(a, retirement_age, return_sequence=None):
    """
    Simulate the portfolio from current_age to the end of the planning horizon.

    When a["has_spouse"] is True the horizon extends to the later of the two
    life expectancies. SS income, expenses, and withdrawal behaviour all
    adjust as each person retires or dies.

    Parameters
    ----------
    a : dict  — the assumptions dictionary
    retirement_age : int  — self's retirement age
    return_sequence : list[float], optional

    Returns
    -------
    pandas.DataFrame with one row per simulated year (indexed by self's age).
    """
    has_spouse = a.get("has_spouse", False)
    self_years = a["life_expectancy"] - a["current_age"]
    if has_spouse:
        spouse_years = a["spouse_life_expectancy"] - a["spouse_current_age"]
        years = max(self_years, spouse_years)
    else:
        years = self_years

    balance = a["current_savings"]
    contribution = a["annual_contribution"]
    rows = []

    for i in range(years):
        age = a["current_age"] + i

        if has_spouse:
            spouse_age = a["spouse_current_age"] + i
            self_alive   = i < self_years
            spouse_alive = i < spouse_years
            if not self_alive and not spouse_alive:
                break
            spouse_retired = spouse_age >= a["spouse_retirement_age"]
        else:
            spouse_age    = 0
            self_alive    = True
            spouse_alive  = False
            spouse_retired = False

        self_retired = age >= retirement_age

        # Returns: switch to post-retirement as soon as either person retires
        if return_sequence is not None:
            r = return_sequence[i]
        else:
            r = (a["post_retirement_return"]
                 if (self_retired or spouse_retired)
                 else a["pre_retirement_return"])

        # ------------------------------------------------------------------
        # SS income
        # ------------------------------------------------------------------
        if has_spouse:
            self_ss_rate   = a["social_security_monthly"] * 12
            spouse_ss_rate = a["spouse_ss_monthly"] * 12
            self_ss_active = self_ss_rate if age >= a["social_security_start_age"] else 0
            spouse_ss_active = (
                spouse_ss_rate if spouse_age >= a["spouse_ss_start_age"] else 0
            )

            if self_alive and spouse_alive:
                ss_income = self_ss_active + spouse_ss_active
            elif self_alive and not spouse_alive:
                # Simplified survivor rule: the survivor receives the higher
                # active benefit. The deceased spouse's amount is available
                # only if it had started before death. This avoids granting a
                # full unclaimed future benefit without modeling survivor
                # claiming ages and reductions.
                spouse_survivor_ss = (
                    spouse_ss_rate
                    if a["spouse_life_expectancy"] > a["spouse_ss_start_age"]
                    else 0
                )
                ss_income = max(self_ss_active, spouse_survivor_ss)
            elif spouse_alive and not self_alive:
                self_survivor_ss = (
                    self_ss_rate
                    if a["life_expectancy"] > a["social_security_start_age"]
                    else 0
                )
                ss_income = max(spouse_ss_active, self_survivor_ss)
            else:
                ss_income = 0
        else:
            ss_income = (
                a["social_security_monthly"] * 12
                if age >= a["social_security_start_age"] else 0
            )

        # Pension income (self only)
        pension_income = (
            a["pension_monthly"] * 12
            if a["pension_start_age"] is not None and age >= a["pension_start_age"]
            else 0
        )

        # Expenses: apply spending curve, then reduce to survivor fraction after first death
        curve_expenses = _spending_curve(a["annual_expenses"], age, a)
        if has_spouse and not (self_alive and spouse_alive):
            effective_expenses = curve_expenses * a["survivor_spending_pct"]
        else:
            effective_expenses = curve_expenses

        balance_start = balance

        # Contributions stop when self retires; portfolio moves to withdrawal
        # mode whenever self has retired OR self has died (spouse still alive)
        in_withdrawal = self_retired or not self_alive
        if in_withdrawal:
            withdrawal = max(effective_expenses - ss_income - pension_income, 0)
            balance -= withdrawal
            contrib_this_year = 0
        else:
            withdrawal = 0
            balance += contribution
            contrib_this_year = contribution
            contribution *= (1 + a["contribution_growth_rate"])

        growth   = balance * r
        balance += growth

        rows.append({
            "age":           age,
            "retired":       self_retired,
            "balance_start": balance_start,
            "contribution":  contrib_this_year,
            "ss_income":     ss_income,
            "pension_income": pension_income,
            "withdrawal":    withdrawal,
            "growth":        growth,
            "balance_end":   balance,
        })

    return pd.DataFrame(rows)


def find_earliest_retirement_age(a, age_range=None):
    if age_range is None:
        age_range = range(a["current_age"] + 1, a["life_expectancy"])

    for age in age_range:
        df = project_portfolio(a, age)
        if (df["balance_end"] >= 0).all():
            return age, df

    return None, None


def monte_carlo_success(a, retirement_age, n_sims=1000, seed=42, return_paths=False):
    """
    Run n_sims randomized projections for a given retirement age.

    Returns
    -------
    success_rate : float
    ending_balances : list[float]
    paths : np.ndarray, shape (n_sims, years)   — only when return_paths=True
    """
    rng = np.random.default_rng(seed)

    self_years = a["life_expectancy"] - a["current_age"]
    if a.get("has_spouse", False):
        spouse_years   = a["spouse_life_expectancy"] - a["spouse_current_age"]
        years          = max(self_years, spouse_years)
        spouse_ret_age = a["spouse_retirement_age"]
    else:
        years          = self_years
        spouse_ret_age = None

    successes       = 0
    ending_balances = []
    all_paths       = [] if return_paths else None

    for _ in range(n_sims):
        returns = []
        for i in range(years):
            age          = a["current_age"] + i
            anyone_retired = age >= retirement_age or (
                spouse_ret_age is not None and
                a["spouse_current_age"] + i >= spouse_ret_age
            )
            mean_r = (a["post_retirement_return"] if anyone_retired
                      else a["pre_retirement_return"])
            returns.append(rng.normal(mean_r, a["return_volatility"]))

        df = project_portfolio(a, retirement_age, return_sequence=returns)
        ending_balances.append(df["balance_end"].iloc[-1])
        if (df["balance_end"] >= 0).all():
            successes += 1
        if return_paths:
            all_paths.append(df["balance_end"].values)

    if return_paths:
        return successes / n_sims, ending_balances, np.array(all_paths)
    return successes / n_sims, ending_balances


def mc_success_grid(a, ages, balances, n_sims=300, seed=42,
                    inflation_rate=0.0, base_age=None):
    """
    Vectorised success-rate grid over retirement ages × starting balances.
    Each cell answers: "if I retire at `age` with `balance`, what fraction
    of simulations survive to life expectancy?"
    By default, `balances` are today's dollars. If inflation_rate > 0 and
    base_age is set, callers may pass nominal balances, which are converted
    to real dollars for each age column. All years use post_retirement_return
    (person is already retired).
    """
    rng  = np.random.default_rng(seed)
    grid = np.zeros((len(balances), len(ages)))

    for j, age in enumerate(ages):
        years = a["life_expectancy"] - age
        if years <= 0:
            grid[:, j] = 1.0
            continue
        ret = rng.normal(a["post_retirement_return"], a["return_volatility"], (n_sims, years))

        withdrawals = np.zeros(years)
        for yr in range(years):
            curr_age = age + yr
            ss = a["social_security_monthly"] * 12 if curr_age >= a["social_security_start_age"] else 0
            pension = (
                a["pension_monthly"] * 12
                if a["pension_start_age"] is not None and curr_age >= a["pension_start_age"]
                else 0
            )
            withdrawals[yr] = max(_spending_curve(a["annual_expenses"], curr_age, a) - ss - pension, 0)

        if inflation_rate > 0 and base_age is not None:
            real_bals = [b / (1 + inflation_rate) ** (age - base_age) for b in balances]
        else:
            real_bals = balances

        for i, bal in enumerate(real_bals):
            b     = np.full(n_sims, float(bal))
            alive = np.ones(n_sims, dtype=bool)
            for yr in range(years):
                b -= withdrawals[yr]
                b *= (1 + ret[:, yr])
                alive &= (b >= 0)
            grid[i, j] = alive.mean()

    return grid


In [ ]:
_style = {"description_width": "190px"}
_layout         = Layout(width="340px")                    # always-visible widgets
_spouse_layout    = Layout(width="340px", display="none")  # toggled by Has spouse?
_spouse_ss_layout = Layout(width="340px", display="none")  # toggled by Has spouse?
_pension_layout   = Layout(width="340px", display="none")  # toggled by Has pension?
_threephase_layout = Layout(width="340px", display="none") # toggled by spending model
_taper_layout      = Layout(width="340px", display="none") # toggled by spending model

# --- Personal ---
w_current_age = widgets.BoundedIntText(
    value=56, min=18, max=90,
    description="Current age:", style=_style, layout=_layout)
w_target_retirement_age = widgets.BoundedIntText(
    value=65, min=40, max=80,
    description="Target ret. age:", style=_style, layout=_layout)
w_life_expectancy = widgets.BoundedIntText(
    value=90, min=70, max=110,
    description="Life expectancy:", style=_style, layout=_layout)

# --- Spouse (gated) ---
w_has_spouse = widgets.ToggleButton(
    value=False, description="Has spouse?",
    button_style="", layout=_layout)
w_spouse_age = widgets.BoundedIntText(
    value=54, min=18, max=90,
    description="Spouse age:", style=_style, layout=_spouse_layout)
w_spouse_life_expectancy = widgets.BoundedIntText(
    value=92, min=70, max=110,
    description="Spouse life exp.:", style=_style, layout=_spouse_layout)
w_spouse_retirement_age = widgets.BoundedIntText(
    value=63, min=40, max=80,
    description="Spouse ret. age:", style=_style, layout=_spouse_layout)
w_survivor_spending_pct = widgets.BoundedFloatText(
    value=70.0, min=50.0, max=100.0, step=5.0,
    description="Survivor spending (%):", style=_style, layout=_spouse_layout)

# --- Savings ---
w_current_savings = widgets.BoundedFloatText(
    value=250, min=0, max=1e6, step=10,
    description="Current savings ($k):", style=_style, layout=_layout)
w_annual_contribution = widgets.BoundedFloatText(
    value=15, min=0, max=500, step=1,
    description="Annual contribution ($k):", style=_style, layout=_layout)
w_contribution_growth_rate = widgets.BoundedFloatText(
    value=2.0, min=0, max=10.0, step=0.5,
    description="Contrib. growth (real %):", style=_style, layout=_layout)

# --- Returns (enter nominal; inflation is subtracted internally) ---
w_inflation_rate = widgets.BoundedFloatText(
    value=2.5, min=0.0, max=10.0, step=0.5,
    description="Inflation rate (%):", style=_style, layout=_layout)
w_pre_retirement_return = widgets.BoundedFloatText(
    value=7.5, min=0, max=20.0, step=0.5,
    description="Pre-ret. return (nominal %):", style=_style, layout=_layout)
w_post_retirement_return = widgets.BoundedFloatText(
    value=5.5, min=0, max=15.0, step=0.5,
    description="Post-ret. return (nominal %):", style=_style, layout=_layout)
w_return_volatility = widgets.BoundedFloatText(
    value=12.0, min=0, max=30.0, step=0.5,
    description="Return volatility (%):", style=_style, layout=_layout)

# --- Spending / Social Security ---
w_annual_expenses = widgets.BoundedFloatText(
    value=60, min=0, max=1e4, step=1,
    description="Annual expenses ($k):", style=_style, layout=_layout)

# Spending model selector (Dropdown keeps long labels off-screen until opened)
w_spending_model = widgets.Dropdown(
    options=[("Flat (constant)", "flat"),
             ("Three-phase (go-go/slow-go/no-go)", "three_phase"),
             ("Annual taper", "taper")],
    value="flat", description="Spending model:", style=_style, layout=_layout)

w_slow_go_age = widgets.BoundedIntText(
    value=75, min=60, max=95,
    description="Slow-go starts (age):", style=_style, layout=_threephase_layout)
w_slow_go_pct = widgets.BoundedFloatText(
    value=80.0, min=10.0, max=100.0, step=5.0,
    description="Slow-go spending (%):", style=_style, layout=_threephase_layout)
w_no_go_age = widgets.BoundedIntText(
    value=85, min=65, max=100,
    description="No-go starts (age):", style=_style, layout=_threephase_layout)
w_no_go_pct = widgets.BoundedFloatText(
    value=60.0, min=10.0, max=100.0, step=5.0,
    description="No-go spending (%):", style=_style, layout=_threephase_layout)

w_taper_start_age = widgets.BoundedIntText(
    value=75, min=60, max=95,
    description="Taper starts (age):", style=_style, layout=_taper_layout)
w_taper_rate_pct = widgets.BoundedFloatText(
    value=1.5, min=0.1, max=10.0, step=0.5,
    description="Taper rate (%/yr):", style=_style, layout=_taper_layout)

w_social_security_monthly = widgets.BoundedFloatText(
    value=1.8, min=0, max=50, step=0.1,
    description="SS monthly ($k):", style=_style, layout=_layout)
w_social_security_start_age = widgets.RadioButtons(
    options=[62, 65, 67, 70], value=67,
    description="SS start age:", style=_style, layout=_layout)

# Spouse SS (gated)
w_spouse_ss_monthly = widgets.BoundedFloatText(
    value=1.5, min=0, max=50, step=0.1,
    description="Spouse SS monthly ($k):", style=_style, layout=_spouse_ss_layout)
w_spouse_ss_start_age = widgets.RadioButtons(
    options=[62, 65, 67, 70], value=67,
    description="Spouse SS age:", style=_style, layout=_spouse_ss_layout)

# --- Pension (gated) ---
w_has_pension = widgets.ToggleButton(
    value=False, description="Has pension?",
    button_style="", layout=_layout)
w_pension_monthly = widgets.BoundedFloatText(
    value=0, min=0, max=50, step=0.1,
    description="Pension monthly ($k):", style=_style, layout=_pension_layout)
w_pension_start_age = widgets.BoundedIntText(
    value=60, min=40, max=80,
    description="Pension start age:", style=_style, layout=_pension_layout)

# --- Monte Carlo controls ---
w_n_sims = widgets.RadioButtons(
    options=[500, 1000, 2000, 5000], value=1000,
    description="# simulations:", style=_style, layout=_layout)
run_mc_button = widgets.Button(
    description="Run Monte Carlo", button_style="primary", layout=_layout)
run_grid_button = widgets.Button(
    description="Run Balance Grid", button_style="info", layout=_layout)

# --- Scenario save/load ---
w_scenario_name = widgets.Text(
    value="scenario", description="Scenario name:",
    style=_style, layout=_layout)
save_scenario_button = widgets.Button(
    description="Save", button_style="", layout=Layout(width="164px"))
load_upload = widgets.FileUpload(
    accept=".json", multiple=False,
    description="Load", layout=Layout(width="164px"))
scenario_status = widgets.HTML(value="")


def _toggle_spouse(change=None):
    display_val = "" if w_has_spouse.value else "none"
    _spouse_layout.display    = display_val
    _spouse_ss_layout.display = display_val


def _toggle_pension(change=None):
    _pension_layout.display = "" if w_has_pension.value else "none"


def _toggle_spending_model(change=None):
    model = w_spending_model.value
    _threephase_layout.display = "" if model == "three_phase" else "none"
    _taper_layout.display      = "" if model == "taper"       else "none"


w_has_spouse.observe(_toggle_spouse, names="value")
w_has_pension.observe(_toggle_pension, names="value")
w_spending_model.observe(_toggle_spending_model, names="value")


def build_assumptions_from_widgets():
    inflation = w_inflation_rate.value / 100
    # Convert nominal returns to real using the Fisher equation:
    #   real = (1 + nominal) / (1 + inflation) - 1
    pre_ret_real  = (1 + w_pre_retirement_return.value  / 100) / (1 + inflation) - 1
    post_ret_real = (1 + w_post_retirement_return.value / 100) / (1 + inflation) - 1
    return {
        "current_age":             w_current_age.value,
        "life_expectancy":         w_life_expectancy.value,
        "current_savings":         w_current_savings.value * 1_000,
        "annual_contribution":     w_annual_contribution.value * 1_000,
        "contribution_growth_rate": w_contribution_growth_rate.value / 100,
        "pre_retirement_return":   pre_ret_real,
        "post_retirement_return":  post_ret_real,
        "return_volatility":       w_return_volatility.value / 100,
        "annual_expenses":         w_annual_expenses.value * 1_000,
        "spending_model":          w_spending_model.value,
        "spending_params": {
            "slow_go_age":     w_slow_go_age.value,
            "slow_go_pct":     w_slow_go_pct.value / 100,
            "no_go_age":       w_no_go_age.value,
            "no_go_pct":       w_no_go_pct.value / 100,
            "taper_start_age": w_taper_start_age.value,
            "taper_rate":      w_taper_rate_pct.value / 100,
        },
        "social_security_monthly": w_social_security_monthly.value * 1_000,
        "social_security_start_age": w_social_security_start_age.value,
        "pension_monthly":         w_pension_monthly.value * 1_000 if w_has_pension.value else 0,
        "pension_start_age":       w_pension_start_age.value if w_has_pension.value else None,
        "target_retirement_age":   w_target_retirement_age.value,
        # Spouse
        "has_spouse":              w_has_spouse.value,
        "spouse_current_age":      w_spouse_age.value,
        "spouse_life_expectancy":  w_spouse_life_expectancy.value,
        "spouse_retirement_age":   w_spouse_retirement_age.value,
        "survivor_spending_pct":   w_survivor_spending_pct.value / 100,
        "spouse_ss_monthly":       w_spouse_ss_monthly.value * 1_000,
        "spouse_ss_start_age":     w_spouse_ss_start_age.value,
    }


In [ ]:

# Prevent flex compression: VBox children must keep their natural height.
# Without this, the sidebar squishes bottom items when content exceeds 100vh.
from IPython.display import HTML as _IPyHTML
display(_IPyHTML("<style>.widget-vbox > * { flex-shrink: 0 !important; }</style>"))

# --- Output widgets ---
output_stats          = widgets.Output(layout=Layout(width="100%"))
output_projection_plot = widgets.Output(layout=Layout(width="100%"))
output_earliest_age   = widgets.Output(layout=Layout(width="100%"))
output_mc_stats       = widgets.Output(layout=Layout(width="100%"))
output_mc_fan         = widgets.Output(layout=Layout(width="100%"))
output_sweep_plot     = widgets.Output(layout=Layout(width="100%"))
output_sweep_table    = widgets.Output(layout=Layout(width="100%"))
output_grid_plot      = widgets.Output(layout=Layout(width="100%"))

# --- Sidebar ---
# All widgets live directly in this VBox — no inner containers — so there is
# exactly one scrollbar (overflow_y=auto on the sidebar itself).
# Toggleable groups (spouse / pension / spending model) are hidden by setting
# display="none" on their shared Layout objects in widget-defs.
sidebar = widgets.VBox([
    widgets.HTML("<b>Personal</b>"),
    w_current_age,
    w_life_expectancy,
    w_target_retirement_age,
    w_has_spouse,
    w_spouse_age,
    w_spouse_life_expectancy,
    w_spouse_retirement_age,
    w_survivor_spending_pct,
    widgets.HTML("<b>Savings</b>"),
    w_current_savings,
    w_annual_contribution,
    w_contribution_growth_rate,
    widgets.HTML("<b>Returns (nominal rates)</b>"),
    w_inflation_rate,
    w_pre_retirement_return,
    w_post_retirement_return,
    w_return_volatility,
    widgets.HTML("<b>Spending / Social Security</b>"),
    w_annual_expenses,
    w_spending_model,
    w_slow_go_age, w_slow_go_pct, w_no_go_age, w_no_go_pct,
    w_taper_start_age, w_taper_rate_pct,
    w_social_security_monthly,
    w_social_security_start_age,
    w_spouse_ss_monthly,
    w_spouse_ss_start_age,
    widgets.HTML("<b>Pension</b>"),
    w_has_pension,
    w_pension_monthly,
    w_pension_start_age,
    widgets.HTML("<b>Monte Carlo</b>"),
    w_n_sims,
    run_mc_button,
    run_grid_button,
    widgets.HTML("<b>Scenario</b>"),
    w_scenario_name,
    widgets.HBox([save_scenario_button, load_upload], layout=Layout(width="340px")),
    scenario_status,
], layout=Layout(
    min_width="365px", max_width="365px",
    overflow_y="auto",
    overflow_x="hidden",
    padding="12px",
))

# --- Main area ---
main_area = widgets.VBox([
    widgets.HTML("<h3>Deterministic Projection</h3>"),
    output_stats,
    output_projection_plot,
    widgets.HTML("<h3>Earliest Retirement Age</h3>"),
    output_earliest_age,
    widgets.HTML("<h3>Monte Carlo Results</h3>"),
    output_mc_stats,
    output_mc_fan,
    widgets.HTML("<h3>Success Rate vs. Retirement Age</h3>"),
    output_sweep_plot,
    output_sweep_table,
    widgets.HTML("<h3>Success Rate: Age × Balance</h3>"),
    output_grid_plot,
], layout=Layout(flex="1", overflow_y="auto", padding="8px"))

# --- Input Guide area ---
_guide_html = """
<div style="font-family: sans-serif; font-size: 13px; line-height: 1.6; max-width: 820px; padding: 4px 12px;">

<h2 style="margin-top:8px">Input Guide</h2>
<p style="color:#555">Enter dollar amounts in <strong>today's dollars</strong>
(expenses, savings, Social Security). Enter rates of return as the
<strong>nominal percentage you see on your brokerage or financial website</strong> —
the model subtracts inflation internally.</p>

<hr/>

<h3 style="color:#2471a3">Personal</h3>

<h4>Current Age</h4>
<p>Your age right now. The model simulates every year from this age to your life expectancy.</p>

<h4>Life Expectancy</h4>
<p>How long your money needs to last. This is the planning horizon, not a prediction.
Using a longer value is conservative — it forces the plan to hold up longer.</p>
<ul>
  <li><strong>Rough guide:</strong> Social Security Administration period life tables show a 65-year-old
      man can expect to live to ~84, a woman to ~87. But half of all people live <em>longer</em>
      than the median.</li>
  <li><strong>Practical choices:</strong> 90 is a common conservative target; 95 if longevity runs in
      your family or you want extra margin.</li>
  <li>Visit <a href="https://www.ssa.gov/oact/population/longevity.html" target="_blank">ssa.gov/oact/population/longevity.html</a>
      for the official life expectancy calculator.</li>
</ul>

<h4>Target Retirement Age</h4>
<p>The age at which you plan to stop working and start drawing down the portfolio.
The <em>Earliest Retirement Age</em> panel tells you the earliest age the deterministic
projection stays solvent — use that as a sanity check on your target.</p>

<h4>Spouse</h4>
<p>Toggle <em>Has spouse?</em> to add a second person to the model. The planning horizon
automatically extends to the later of the two life expectancies. Key effects:</p>
<ul>
  <li><strong>Retirement age</strong> — returns switch to the post-retirement rate as soon as
      <em>either</em> person retires (more conservative).</li>
  <li><strong>SS income</strong> — both Social Security streams are active while each person
      is alive and past their claiming age. When one spouse dies, the survivor keeps
      the higher active benefit. This is an approximation; actual survivor benefits depend
      on age, claiming history, and reductions.</li>
  <li><strong>Survivor spending</strong> — after the first death, expenses drop to the fraction
      you specify (typically 60–75% of joint spending).</li>
  <li><strong>Contributions</strong> — the annual contribution field represents your household
      total and grows by the real percentage you enter. It stops when <em>you</em> retire; the model
      does not estimate wage income from either spouse.</li>
</ul>

<hr/>

<h3 style="color:#2471a3">Savings</h3>

<h4>Current Savings ($k)</h4>
<p>Total investable assets today: 401(k), IRA, taxable brokerage, etc.
<em>Do not include</em> home equity, car value, or cash you plan to spend soon.
Enter the number in thousands (e.g. 500 for $500,000).</p>

<h4>Annual Contribution ($k)</h4>
<p>How much you add to the portfolio each year, in today's dollars.
Include employer match if it goes into an investment account.
For 2026, the 401(k)/403(b)/457/TSP employee limit is $24,500; IRA limit is $7,500.</p>

<h4>Contribution Growth Rate (real %)</h4>
<p>How fast your annual contribution grows each year in real terms (above inflation).
If your nominal savings dollar amount rises only with inflation, enter 0.
If your savings rate is genuinely increasing — e.g. a mortgage pays off soon — 1–3% is reasonable.</p>

<hr/>

<h3 style="color:#2471a3">Returns (nominal rates)</h3>

<p style="background:#eaf4fb; padding:8px; border-left:4px solid #2471a3; border-radius:3px">
Enter the <strong>nominal rates of return you see on your brokerage or financial website</strong>
(e.g. 10%). Set your expected inflation rate once — the model converts to real
(inflation-adjusted) returns internally using the Fisher equation:
<em>real = (1 + nominal) / (1 + inflation) − 1</em>.
All balances and projections are displayed in today's purchasing power.</p>

<h4>Inflation Rate (%)</h4>
<p>Your assumed long-run inflation rate. Set it once and leave it alone.
Common choices:</p>
<ul>
  <li><strong>2.5%</strong> — near the Fed's long-run target; reasonable default.</li>
  <li><strong>3%</strong> — slightly more conservative; near the historical average (1926–2023).</li>
  <li><strong>3.5–4%</strong> — pessimistic / stress-test scenario.</li>
</ul>

<h4>Pre-Retirement Return (nominal %)</h4>
<p>The nominal annual return you expect while still working — the number reported
on a financial website or fund fact sheet. Typical ranges by allocation:</p>
<table style="border-collapse:collapse; font-size:12px; margin-bottom:8px">
  <tr style="background:#d6eaf8">
    <th style="padding:4px 10px; text-align:left">Allocation</th>
    <th style="padding:4px 10px; text-align:left">Nominal return range</th>
    <th style="padding:4px 10px; text-align:left">Volatility range</th>
  </tr>
  <tr><td style="padding:4px 10px">100% stocks</td><td style="padding:4px 10px">8–10%</td><td style="padding:4px 10px">15–18%</td></tr>
  <tr style="background:#eaf4fb"><td style="padding:4px 10px">80/20 stocks/bonds</td><td style="padding:4px 10px">7–9%</td><td style="padding:4px 10px">12–15%</td></tr>
  <tr><td style="padding:4px 10px">60/40 stocks/bonds</td><td style="padding:4px 10px">6–7.5%</td><td style="padding:4px 10px">9–12%</td></tr>
  <tr style="background:#eaf4fb"><td style="padding:4px 10px">40/60 stocks/bonds</td><td style="padding:4px 10px">4.5–6%</td><td style="padding:4px 10px">7–10%</td></tr>
  <tr><td style="padding:4px 10px">100% bonds</td><td style="padding:4px 10px">3–5%</td><td style="padding:4px 10px">4–7%</td></tr>
</table>
<p style="font-size:12px;color:#555">Based on historical U.S. data (Ibbotson, 1926–2023).
Past performance does not guarantee future results.</p>

<h4>Post-Retirement Return (nominal %)</h4>
<p>Expected nominal return after you retire. Most people shift to a more conservative
allocation to reduce sequence-of-returns risk — typically 1–2% lower than the
pre-retirement rate.</p>

<h4>Return Volatility (%)</h4>
<p>Standard deviation of annual returns used in Monte Carlo simulations.
Match this to your pre-retirement allocation from the table above.</p>

<hr/>

<h3 style="color:#2471a3">Spending / Social Security</h3>

<h4>Annual Expenses ($k)</h4>
<p>Total annual spending in retirement, in today's dollars. Two common approaches:</p>
<ul>
  <li><strong>Bottom-up:</strong> List expected categories — housing, food, transport, healthcare,
      travel, utilities, insurance — and add them up.</li>
  <li><strong>Rule of thumb:</strong> Many planners use 70–80% of pre-retirement gross income.</li>
</ul>
<p>Don't forget: Medicare Part B premiums (~$185/month in 2025), supplemental insurance,
out-of-pocket costs, and potential long-term care.</p>

<h4>Spending Model</h4>
<p>Real spending in retirement is not constant. Research (e.g. the "smile" pattern)
suggests it often declines in real terms as people age. Choose from three models:</p>
<ul>
  <li><strong>Flat (constant)</strong> — baseline; annual expenses stay the same in real terms
      throughout retirement. Most conservative on the spending side.</li>
  <li><strong>Three-phase (go-go / slow-go / no-go)</strong> — models the activity lifecycle of
      retirement:<br>
      <em>Go-go</em> (early retirement): full spending.<br>
      <em>Slow-go</em> (middle): spending drops to the percentage you specify (default 80%)
        at the age you choose (default 75).<br>
      <em>No-go</em> (late): spending drops again (default 60%) starting at a second age
        (default 85). Travel and entertainment fall off but healthcare costs may rise —
        this model nets those out conservatively.</li>
  <li><strong>Annual taper</strong> — spending declines by a fixed real percentage each year
      after a chosen starting age (e.g. 1.5%/yr after 75). After 10 years at 1.5%/yr,
      spending is about 86% of the base; after 20 years, about 74%.</li>
</ul>
<p style="background:#fef9e7; padding:8px; border-left:4px solid #f39c12; border-radius:3px; font-size:12px">
<strong>Note:</strong> The spending model applies to <em>gross</em> expenses before subtracting
Social Security and pension. Survivor spending (when a spouse dies) is applied on top of the
age-adjusted amount.</p>

<h4>Social Security Monthly ($k)</h4>
<p>Your estimated monthly Social Security benefit at the claiming age you select, in today's dollars.
The most accurate source is <a href="https://www.ssa.gov/myaccount/" target="_blank">ssa.gov/myaccount</a>
— your Statement shows estimated benefits at 62, FRA, and 70.</p>

<h4>Social Security Start Age</h4>
<ul>
  <li><strong>62:</strong> Earliest eligible; benefit reduced ~25–30% vs. FRA permanently.</li>
  <li><strong>67 (Full Retirement Age for those born 1960+):</strong> "Full" benefit.</li>
  <li><strong>70:</strong> Maximum — increases ~8%/year from FRA to 70.</li>
</ul>

<h4>Spouse SS</h4>
<p>Visible when <em>Has spouse?</em> is enabled. When one spouse dies, the survivor keeps
the higher active benefit. This is an approximation; actual survivor benefits depend on age,
claiming history, and reductions.</p>

<hr/>

<h3 style="color:#2471a3">Pension</h3>
<p>Enable if you have a defined-benefit pension. Enter the expected monthly payment in today's
dollars and the age at which it starts.</p>

<hr/>

<h3 style="color:#2471a3">Monte Carlo</h3>
<ul>
  <li><strong>500:</strong> Fast; good for exploring. Success rates accurate to ±2–3%.</li>
  <li><strong>1000:</strong> Default; good balance of speed and accuracy.</li>
  <li><strong>2000–5000:</strong> Use when stress-testing a final plan.</li>
</ul>
<p>The fan chart shows individual paths plus 10/25/75/90th percentile bands.
Red paths are simulations where the portfolio was depleted.</p>

<hr/>

<h3 style="color:#2471a3">Scenario Save / Load</h3>
<p>Use <strong>Save</strong> to write all current inputs to a JSON file.
Use <strong>Load</strong> to restore a previously saved scenario — all widgets update automatically.</p>

</div>
"""

help_area = widgets.HTML(
    value=_guide_html,
    layout=Layout(flex="1", overflow_y="auto", padding="8px")
)

# --- Tab: Dashboard | Input Guide ---
content_tabs = widgets.Tab(
    children=[main_area, help_area],
    layout=Layout(flex="1", overflow_y="auto")
)
content_tabs.set_title(0, "Dashboard")
content_tabs.set_title(1, "Input Guide")

app = widgets.HBox(
    [sidebar, content_tabs],
    layout=Layout(width="100%", height="100vh", overflow="hidden")
)
display(app)


In [ ]:
all_det_widgets = [
    w_current_age, w_life_expectancy, w_current_savings, w_annual_contribution,
    w_contribution_growth_rate, w_inflation_rate,
    w_pre_retirement_return, w_post_retirement_return,
    w_return_volatility, w_annual_expenses, w_social_security_monthly,
    w_social_security_start_age, w_has_pension, w_pension_monthly, w_pension_start_age,
    w_target_retirement_age,
    w_has_spouse, w_spouse_age, w_spouse_life_expectancy, w_spouse_retirement_age,
    w_survivor_spending_pct, w_spouse_ss_monthly, w_spouse_ss_start_age,
    w_spending_model,
    w_slow_go_age, w_slow_go_pct, w_no_go_age, w_no_go_pct,
    w_taper_start_age, w_taper_rate_pct,
]


def update_deterministic(change=None):
    a = build_assumptions_from_widgets()
    proj = project_portfolio(a, a["target_retirement_age"])
    depleted = proj.loc[proj["balance_end"] < 0, "age"]
    retire_idx = proj["retired"].idxmax()
    balance_at_retirement = (
        proj.loc[retire_idx - 1, "balance_end"] if retire_idx > 0 else a["current_savings"]
    )

    with output_stats:
        output_stats.clear_output(wait=True)
        inflation   = w_inflation_rate.value
        pre_nom     = w_pre_retirement_return.value
        post_nom    = w_post_retirement_return.value
        pre_real    = (1 + pre_nom  / 100) / (1 + inflation / 100) - 1
        post_real   = (1 + post_nom / 100) / (1 + inflation / 100) - 1
        print(f"Returns:  pre-ret {pre_nom:.1f}% nominal → {pre_real*100:.2f}% real  |  "
              f"post-ret {post_nom:.1f}% nominal → {post_real*100:.2f}% real  "
              f"(inflation {inflation:.1f}%)")
        print(f"Retirement age tested: {a['target_retirement_age']}")
        if a["has_spouse"]:
            print(f"Spouse retirement age: {a['spouse_retirement_age']}")
            print(f"Planning horizon: your age {a['life_expectancy']} / spouse age {a['spouse_life_expectancy']}")
        print(f"Projected balance at retirement: ${balance_at_retirement/1e3:,.1f}k")
        print(f"Projected balance at end of horizon: ${proj['balance_end'].iloc[-1]/1e3:,.1f}k")
        if not depleted.empty:
            print(f"WARNING: Portfolio runs out at age {int(depleted.iloc[0])}")
        else:
            print("Portfolio lasts through the full planning horizon.")

    with output_projection_plot:
        output_projection_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(proj["age"] + 1, proj["balance_end"], label="Portfolio balance")
        ax.axvline(a["target_retirement_age"], color="gray", linestyle="--",
                   label=f"You retire at {a['target_retirement_age']}")
        if a["has_spouse"]:
            ax.axvline(a["spouse_retirement_age"], color="steelblue", linestyle=":",
                       label=f"Spouse retires at {a['spouse_retirement_age']}")
        ax.axhline(0, color="red", linewidth=0.8)
        ax.set_xlabel("Your age")
        ax.set_ylabel("Balance ($k, today's dollars)")
        ax.set_title("Projected Portfolio Balance")
        ax.legend()
        ax.grid(True)
        ax.yaxis.set_major_formatter(lambda x, _: f"${x/1e3:,.0f}k")
        plt.tight_layout()
        plt.show()
        plt.close("all")

    with output_earliest_age:
        output_earliest_age.clear_output(wait=True)
        earliest, earliest_df = find_earliest_retirement_age(a)
        if earliest is not None:
            print(f"Earliest sustainable retirement age (average scenario): {earliest}")
            print(f"Projected balance at end of horizon: ${earliest_df['balance_end'].iloc[-1]/1e3:,.1f}k")
        else:
            print("No retirement age in the tested range fully sustains your spending.")
            print("Consider increasing savings/contributions, lowering expenses, or extending the age range.")


def run_monte_carlo(btn=None):
    run_mc_button.disabled = True
    run_mc_button.description = "Running..."
    try:
        a = build_assumptions_from_widgets()
        rate, balances, paths = monte_carlo_success(
            a, a["target_retirement_age"], n_sims=w_n_sims.value, return_paths=True)

        with output_mc_stats:
            output_mc_stats.clear_output(wait=True)
            print(f"Retirement age tested: {a['target_retirement_age']}")
            print(f"Success rate (portfolio never depleted): {rate:.1%}")
            print(f"Median ending balance:          ${np.median(balances)/1e3:,.1f}k")
            print(f"10th percentile ending balance: ${np.percentile(balances, 10)/1e3:,.1f}k")
            print(f"90th percentile ending balance: ${np.percentile(balances, 90)/1e3:,.1f}k")

        with output_mc_fan:
            output_mc_fan.clear_output(wait=True)
            total_years = paths.shape[1]
            ages = np.arange(a["current_age"] + 1, a["current_age"] + 1 + total_years)
            success_mask = (paths >= 0).all(axis=1)
            n_sims_run = len(paths)

            fig, ax = plt.subplots(figsize=(9, 5))

            rng_vis = np.random.default_rng(0)
            n_plot = min(300, n_sims_run)
            plot_idx = rng_vis.choice(n_sims_run, n_plot, replace=False)
            for i in plot_idx:
                clr = "#e74c3c" if not success_mask[i] else "#3498db"
                ax.plot(ages, paths[i] / 1e3, color=clr, alpha=0.04, linewidth=0.5)

            pcts = np.percentile(paths, [10, 25, 50, 75, 90], axis=0)

            ax.fill_between(ages, pcts[0] / 1e3, pcts[4] / 1e3,
                            color="#aed6f1", alpha=0.7, label="10–90th pct")
            ax.plot(ages, pcts[0] / 1e3, color="#5dade2", linewidth=1.0, linestyle="--")
            ax.plot(ages, pcts[4] / 1e3, color="#5dade2", linewidth=1.0, linestyle="--")

            ax.fill_between(ages, pcts[1] / 1e3, pcts[3] / 1e3,
                            color="#2e86c1", alpha=0.5, label="25–75th pct")
            ax.plot(ages, pcts[1] / 1e3, color="#1a5276", linewidth=1.0, linestyle="-")
            ax.plot(ages, pcts[3] / 1e3, color="#1a5276", linewidth=1.0, linestyle="-")

            ax.plot(ages, pcts[2] / 1e3, color="#1a5276", linewidth=2.5, label="Median")

            ax.axvline(a["target_retirement_age"], color="gray", linestyle="--",
                       label=f"You retire at {a['target_retirement_age']}")
            if a["has_spouse"]:
                ax.axvline(a["spouse_retirement_age"], color="steelblue", linestyle=":",
                           label=f"Spouse retires at {a['spouse_retirement_age']}")
            ax.axhline(0, color="red", linewidth=0.8)
            ax.set_xlabel("Your age")
            ax.set_ylabel("Balance ($k, today's dollars)")
            n_failed = (~success_mask).sum()
            ax.set_title(
                f"Monte Carlo paths — retire at {a['target_retirement_age']}  "
                f"({rate:.1%} success,  {n_failed}/{n_sims_run} depleted)")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.yaxis.set_major_formatter(lambda x, _: f"${x:,.0f}k")
            plt.tight_layout()
            plt.show()
            plt.close("all")

        sweep_start = max(a["current_age"] + 1, 50)
        sweep_end = min(a["life_expectancy"], 81)
        sweep_results = []
        for age in range(sweep_start, sweep_end):
            r, _ = monte_carlo_success(a, age, n_sims=500)
            sweep_results.append({"retirement_age": age, "success_rate": r})
        sweep_df = pd.DataFrame(sweep_results)

        with output_sweep_plot:
            output_sweep_plot.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.plot(sweep_df["retirement_age"], sweep_df["success_rate"] * 100, marker="o")
            ax.axhline(90, color="green", linestyle="--", label="90% success")
            ax.axhline(80, color="orange", linestyle="--", label="80% success")
            ax.set_xlabel("Your retirement age")
            ax.set_ylabel("Success rate (%)")
            ax.set_title("Plan Success Rate vs. Retirement Age")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()
            plt.close("all")

        with output_sweep_table:
            output_sweep_table.clear_output(wait=True)
            display(sweep_df.style.format({"success_rate": "{:.1%}"}))

    finally:
        run_mc_button.disabled = False
        run_mc_button.description = "Run Monte Carlo"


run_mc_button.on_click(run_monte_carlo)

for w in all_det_widgets:
    w.observe(update_deterministic, names="value")


def run_balance_grid(btn=None):
    run_grid_button.disabled = True
    run_grid_button.description = "Running..."
    try:
        a = build_assumptions_from_widgets()
        ages = list(range(max(a["current_age"] + 1, 50), min(a["life_expectancy"], 81), 2))
        low  = a["current_savings"] * 0.25
        high = a["current_savings"] * 8.0
        balances = list(np.linspace(low, high, 12))

        grid = mc_success_grid(a, ages, balances, n_sims=500)

        with output_grid_plot:
            output_grid_plot.clear_output(wait=True)
            ages_arr = np.array(ages)
            bals_arr = np.array(balances) / 1e3
            fig, ax = plt.subplots(figsize=(9, 5))
            cf = ax.contourf(ages_arr, bals_arr, grid * 100,
                             levels=np.linspace(0, 100, 21),
                             cmap="RdYlGn", vmin=0, vmax=100)
            plt.colorbar(cf, ax=ax, label="Success rate (%)")
            cs = ax.contour(ages_arr, bals_arr, grid * 100,
                            levels=[80, 90], colors=["black", "black"], linewidths=2)
            ax.clabel(cs, fmt="%d%%", fontsize=9)
            ax.set_xlabel("Your retirement age")
            ax.set_ylabel("Portfolio balance at retirement ($k, today's dollars)")
            ax.set_title("Monte Carlo success rate: retirement age \u00d7 starting balance")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            plt.close("all")
    finally:
        run_grid_button.disabled = False
        run_grid_button.description = "Run Balance Grid"

run_grid_button.on_click(run_balance_grid)


def save_scenario(btn=None):
    data = {
        "current_age":             w_current_age.value,
        "life_expectancy":         w_life_expectancy.value,
        "target_retirement_age":   w_target_retirement_age.value,
        "current_savings":         w_current_savings.value,
        "annual_contribution":     w_annual_contribution.value,
        "contribution_growth_rate": w_contribution_growth_rate.value,
        "inflation_rate":          w_inflation_rate.value,
        "pre_retirement_return":   w_pre_retirement_return.value,
        "post_retirement_return":  w_post_retirement_return.value,
        "return_volatility":       w_return_volatility.value,
        "annual_expenses":         w_annual_expenses.value,
        "spending_model":          w_spending_model.value,
        "slow_go_age":             w_slow_go_age.value,
        "slow_go_pct":             w_slow_go_pct.value,
        "no_go_age":               w_no_go_age.value,
        "no_go_pct":               w_no_go_pct.value,
        "taper_start_age":         w_taper_start_age.value,
        "taper_rate_pct":          w_taper_rate_pct.value,
        "social_security_monthly": w_social_security_monthly.value,
        "social_security_start_age": w_social_security_start_age.value,
        "has_pension":             w_has_pension.value,
        "pension_monthly":         w_pension_monthly.value,
        "pension_start_age":       w_pension_start_age.value,
        "n_sims":                  w_n_sims.value,
        "has_spouse":              w_has_spouse.value,
        "spouse_age":              w_spouse_age.value,
        "spouse_life_expectancy":  w_spouse_life_expectancy.value,
        "spouse_retirement_age":   w_spouse_retirement_age.value,
        "survivor_spending_pct":   w_survivor_spending_pct.value,
        "spouse_ss_monthly":       w_spouse_ss_monthly.value,
        "spouse_ss_start_age":     w_spouse_ss_start_age.value,
    }
    name = w_scenario_name.value.strip() or "scenario"
    filename = name if name.endswith(".json") else name + ".json"
    path = os.path.join(os.getcwd(), filename)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    scenario_status.value = f"<span style='color:green'>Saved to {path}</span>"


def _load_scenario(change):
    if not load_upload.value:
        return
    try:
        file_info = load_upload.value[0]
        data = json.loads(file_info["content"])
        w_scenario_name.value = file_info["name"].replace(".json", "")
        w_current_age.value              = data.get("current_age", w_current_age.value)
        w_life_expectancy.value          = data.get("life_expectancy", w_life_expectancy.value)
        w_target_retirement_age.value    = data.get("target_retirement_age", w_target_retirement_age.value)
        w_current_savings.value          = data.get("current_savings", w_current_savings.value)
        w_annual_contribution.value      = data.get("annual_contribution", w_annual_contribution.value)
        w_contribution_growth_rate.value = data.get("contribution_growth_rate", w_contribution_growth_rate.value)
        w_inflation_rate.value           = data.get("inflation_rate", w_inflation_rate.value)
        w_pre_retirement_return.value    = data.get("pre_retirement_return", w_pre_retirement_return.value)
        w_post_retirement_return.value   = data.get("post_retirement_return", w_post_retirement_return.value)
        w_return_volatility.value        = data.get("return_volatility", w_return_volatility.value)
        w_annual_expenses.value          = data.get("annual_expenses", w_annual_expenses.value)
        w_spending_model.value           = data.get("spending_model", "flat")
        w_slow_go_age.value              = data.get("slow_go_age", w_slow_go_age.value)
        w_slow_go_pct.value              = data.get("slow_go_pct", w_slow_go_pct.value)
        w_no_go_age.value                = data.get("no_go_age", w_no_go_age.value)
        w_no_go_pct.value                = data.get("no_go_pct", w_no_go_pct.value)
        w_taper_start_age.value          = data.get("taper_start_age", w_taper_start_age.value)
        w_taper_rate_pct.value           = data.get("taper_rate_pct", w_taper_rate_pct.value)
        w_social_security_monthly.value  = data.get("social_security_monthly", w_social_security_monthly.value)
        w_social_security_start_age.value = data.get("social_security_start_age", w_social_security_start_age.value)
        w_has_pension.value              = data.get("has_pension", w_has_pension.value)
        w_pension_monthly.value          = data.get("pension_monthly", w_pension_monthly.value)
        w_pension_start_age.value        = data.get("pension_start_age", w_pension_start_age.value)
        w_n_sims.value                   = data.get("n_sims", w_n_sims.value)
        w_has_spouse.value               = data.get("has_spouse", w_has_spouse.value)
        w_spouse_age.value               = data.get("spouse_age", w_spouse_age.value)
        w_spouse_life_expectancy.value   = data.get("spouse_life_expectancy", w_spouse_life_expectancy.value)
        w_spouse_retirement_age.value    = data.get("spouse_retirement_age", w_spouse_retirement_age.value)
        w_survivor_spending_pct.value    = data.get("survivor_spending_pct", w_survivor_spending_pct.value)
        w_spouse_ss_monthly.value        = data.get("spouse_ss_monthly", w_spouse_ss_monthly.value)
        w_spouse_ss_start_age.value      = data.get("spouse_ss_start_age", w_spouse_ss_start_age.value)
        scenario_status.value = f"<span style='color:green'>Loaded {file_info['name']}</span>"
    except Exception as e:
        scenario_status.value = f"<span style='color:red'>Error: {e}</span>"


save_scenario_button.on_click(save_scenario)
load_upload.observe(_load_scenario, names="value")

update_deterministic()  # initial render on load


## Notes & Caveats

For the full theoretical background — including mathematical derivations,
justification of modeling choices, discussion of limitations, and
references to the academic literature — see
<a href="https://github.com/jlconlin/RetirementPlanner/blob/main/THEORY.md" target="_blank"><strong>THEORY.md</strong></a>.
